In [ ]:
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from mpmath.calculus import extrapolation
from torch.ao.nn.quantized.functional import threshold
from scaling.utils import functional_form_L0, fit_parametric_form, functional_form_L0_max, fit_linear_model, sigmoid_functional_form
from scaling.utils import double_sigmoid_functional_form
from itertools import product

sns.set_style("white")

# Data Loading

In [ ]:
base_figure_path = Path.cwd() / "figures" / "nanotabpfn"
base_figure_path.mkdir(parents=True, exist_ok=True)
base_output_path = Path.cwd() / "output" / "nanotabpfn"
base_output_path.mkdir(parents=True, exist_ok=True)


df = pd.read_csv("../data/nanotabpfn_data_preprocessed.csv",index_col=[0, 1])
config_columns = [col for col in df.columns if col.startswith('config/')]
print("config columns: ", config_columns)
loss_columns = ['real_data/nll', 'real_data/roc_auc', 'val/val_loss', 'val/small/val_loss', 'val/medium/val_loss']
loss_columns_extended = ['real_data/nll', 'real_data/roc_auc', 'real_data/accuracy', 'real_data/balanced_accuracy',
                         'val/val_loss', 'val/small/val_loss', 'val/medium/val_loss']


FILTER_FLOPS_MAX = 1.00000000e+16 * 1.05
FILTER_FLOPS_MIN = 1.00000000e+11 * 0.95
extrapolation_df = df[FILTER_FLOPS_MAX < df['total_flops_rounded']]
df = df[(FILTER_FLOPS_MAX > df['total_flops_rounded']) & (df['total_flops_rounded'] > FILTER_FLOPS_MIN)]


In [ ]:
df.head()

# Scaling Trends

In [ ]:
def get_pareto_frontier(
        df: pd.DataFrame,
        x_name="flops",
        y_name="Validation Loss",
        minimization=True,
) -> pd.DataFrame:
    """ Function to compute Pareto over FLOPs.
    """
    # NOTE: strict assumption here that x_name is maximized and y_name is minimized
    df_copy = df.copy()
    if not minimization:
        df_copy[y_name] = - df_copy[y_name]
    df_sorted = df_copy.sort_values(by=[x_name, y_name], ascending=[True, True])
    df_sorted = df_sorted.drop_duplicates(subset=[x_name], keep="first")
    pareto_points = []
    min_loss_so_far = float('inf')
    for _, row in df_sorted.iterrows():
        if row[y_name] < min_loss_so_far:
            pareto_points.append(row)
            min_loss_so_far = row[y_name]

    df_copy = pd.DataFrame(pareto_points)
    if not minimization:
        df_copy[y_name] = - df_copy[y_name]
    return df_copy

def get_optimal_values(
        df: pd.DataFrame,
        x_name="flops",
        y_name="Validation Loss",
        minimization=True,
) -> pd.DataFrame:
    df_copy = df.copy()
    if not minimization:
        df_copy[y_name] = - df_copy[y_name]
    df_sorted = df_copy.sort_values(by=[x_name, y_name], ascending=[True, True])
    df_optimal = df_sorted.drop_duplicates(subset=[x_name], keep="first")
    if not minimization:
        df_optimal[y_name] = - df_optimal[y_name]
    return df_optimal




In [ ]:
import math
import numpy as np

FLOP_VALUES = np.array([
    1.00000000e+11, 1.77827941e+11, 3.16227766e+11, 5.62341325e+11,
    1.00000000e+12, 1.77827941e+12, 3.16227766e+12, 5.62341325e+12,
    1.00000000e+13, 1.77827941e+13, 3.16227766e+13, 5.62341325e+13,
    1.00000000e+14, 1.77827941e+14, 3.16227766e+14, 5.62341325e+14,
    1.00000000e+15, 1.77827941e+15, 3.16227766e+15, 5.62341325e+15,
    1.00000000e+16, 1.77827941e+16, 3.16227766e+16, 5.62341325e+16,
    1.00000000e+17, 1.77827941e+17, 3.16227766e+17, 5.62341325e+17,
    1.00000000e+18, 1.77827941e+18, 3.16227766e+18, 5.62341325e+18,
    1.00000000e+19
])


# Add additional columns to analyse
df['prior_max_cells'] = df['config/max_features'] * df['config/num_datapoints_max']
df['embedding_layer_ratio'] = df['config/model_config.embedding_size'] / df['config/model_config.num_layers']
df['prior_datapoint_feature_ratio'] = df['config/num_datapoints_max'] / df['config/max_features']

flops_column = 'total_flops_rounded'

trend_columns = config_columns + ["parameters", "total_cells"] + ["prior_max_cells", "embedding_layer_ratio",
                                                                  "prior_datapoint_feature_ratio"]
loss_columns = ['real_data/nll', 'real_data/roc_auc', 'val/val_loss', 'val/small/val_loss', 'val/medium/val_loss']

# --- Consistent styling per metric ------------------------------------------------
# The two metric families are encoded twice on purpose: warm hues + solid lines for the
# metrics measured on *real data*, cool hues + dashed lines for the metrics measured on
# the *prior validation* sets. The grouping therefore survives greyscale printing and
# colour-vision deficiency, while the hue still identifies the individual metric.
# (Palette checked for lightness band, chroma floor, and CVD / normal-vision separation.)
from matplotlib.lines import Line2D

LOSS_FAMILIES = {
    "real_data": "Real data",
    "val": "Prior validation",
}

# The fits within one family come out almost coincident, so the line widths taper in
# drawing order: the later, thinner line stays visible as a core inside the earlier one.
LOSS_STYLES = {
    'real_data/nll':       {"color": "#d1121f", "linestyle": "-",  "linewidth": 4.5, "label": "NLL"},
    'real_data/roc_auc':   {"color": "#e07b00", "linestyle": "-",  "linewidth": 1.8, "label": "ROC AUC"},
    'val/val_loss':        {"color": "#2a78d6", "linestyle": "--", "linewidth": 5.0, "label": "val loss"},
    'val/small/val_loss':  {"color": "#1baf7a", "linestyle": "--", "linewidth": 3.0, "label": "val loss (small)"},
    'val/medium/val_loss': {"color": "#4a3aa7", "linestyle": "--", "linewidth": 1.4, "label": "val loss (medium)"},
}


def loss_family(loss_column):
    """'real_data' or 'val' -- the family a loss column belongs to."""
    return loss_column.split("/")[0]


def loss_style(loss_column, **overrides):
    """Style kwargs for a loss column, e.g. ax.plot(x, y, **loss_style(col, alpha=0.8))."""
    style = dict(LOSS_STYLES[loss_column])
    style.update(overrides)
    return style


def family_legends(ax, columns=None, locs=("upper left", "lower right"), **legend_kwargs):
    """Draw one legend box per metric family so the real-data / prior-val split is explicit."""
    columns = loss_columns if columns is None else columns
    kwargs = dict(fontsize="x-small", title_fontsize="small", framealpha=0.9)
    kwargs.update(legend_kwargs)
    for (family, title), loc in zip(LOSS_FAMILIES.items(), locs):
        handles = [
            Line2D([], [], **loss_style(col, linewidth=min(LOSS_STYLES[col]["linewidth"], 3.0)))
            for col in columns if loss_family(col) == family
        ]
        if not handles:
            continue
        ax.add_artist(ax.legend(handles=handles, title=title, loc=loc, **kwargs))

# Fit scaling laws for the individual parts.

In [ ]:
# Scaling laws on loss for the different metrics
param_names = {0: "L0", 1: "a", 2: "alpha"}

initial_grid_min = [
    [0, 0.1, 0.5],
    [0.1, 1, 2, 5, 10, 50, 100],
    [-1, -0.5, 0, 0.5, 1]
]
bounds_min = [
    (1e-6, 1),
    (0, 100),
    (-5, 5)
]

initial_grid_max = [
    [0, 1, 2, 4, 8],
    [-30, -20, -10, -5, -2.5, -1, -0.5, 0],
    [-2, -1, -0.5, -0.25, 0, 0.25, 0.5, 1, 2]
]

initial_grid_sigmoid = [
    [0.85, 0.9, 0.95, 0.99, 0.999],    # a_max   (ceiling)
    [2, 4, 6, 8, 10, 12],              # log10(x0)  (midpoint)
    [0.1, 0.3, 1.0, 3.0, 10.0],        # k       (steepness per decade)
    [0.45, 0.5],                       # a_min   (floor)
]

# Big-prior ROC AUC is S-shaped: a plateau, a sharp rise near 2e13 FLOPs, then a slow rise that levels off.
# double_sigmoid_functional_form: a sharp logistic step plus a broad one, all parameters fitted.
initial_grid_double_sigmoid = [
    [0.45, 0.5],                       # a_min   (floor)
    [0.15, 0.2],                       # h1      (height of the sharp step)
    [13.3],                            # x1      (log10 FLOPs of the sharp step)
    [5, 10],                           # k1      (steepness per decade)
    [0.15, 0.25, 0.35],                # h2      (height of the broad step)
    [12.5, 13.5, 14.5],                # x2      (log10 FLOPs of the broad step)
    [0.3, 0.6, 1.0],                   # k2      (steepness per decade)
]
bounds_double_sigmoid = [(0.3, 0.6), (0, 0.5), (12, 15), (0.5, 50), (0, 0.5), (10, 18), (0.01, 5)]

bounds_max = [
    (0, 10),
    (-40, 0),
    (-5, 5)
]

# Load TabPFN Reference
tabpfn_tabarena_baseline = pd.read_csv("../data/summary_tabpfn.csv", index_col=0).loc["tabarena"]


sns.set_context("talk")


save_path = base_figure_path / "pareto_fitting"
save_path.mkdir(parents=True, exist_ok=True)

for loss_idx, loss_column in enumerate(loss_columns_extended):
    fig, ax = plt.subplots(layout='constrained')

    if loss_column.split("/")[1] in tabpfn_tabarena_baseline.index:
        ax.axhline(y=tabpfn_tabarena_baseline[loss_column.split("/")[1]])

    maximization = "roc_auc" in loss_column or "accuracy" in loss_column
    minimization = not maximization
    pareto = get_pareto_frontier(df, flops_column, loss_column, minimization=minimization)
    pareto_extrapolation = get_optimal_values(extrapolation_df, flops_column, loss_column, minimization=minimization)

    ax.scatter(pareto[flops_column], pareto[loss_column], marker='x')
    ax.scatter(pareto_extrapolation[flops_column], pareto_extrapolation[loss_column], marker='x', color='red')

    if True: #minimization:
        initial_grid = initial_grid_min
        bounds = None
        func_form = functional_form_L0
    else:
        initial_grid = initial_grid_max
        bounds = bounds_max
        func_form = functional_form_L0_max

    X_data = pareto[flops_column].values
    y_data = np.log(pareto[loss_column].values) if minimization else np.log(1-pareto[loss_column].values)

    best_params, best_loss = fit_parametric_form(
        func_form=func_form,
        X_data=X_data,
        y_data=y_data,
        initial_grid=list(product(*initial_grid)),
        bounds=bounds,
        delta=1e-3,
        use_scipy=True,
    )
    print(loss_column)
    print(best_params)
    print("irreducible loss: ", np.exp(best_params[0]) if minimization else 1-np.exp(best_params[0]))

    y_pred = func_form(FLOP_VALUES, best_params, return_log_loss=False)
    y_pred = y_pred if minimization else 1-y_pred

    ax.plot(
        FLOP_VALUES,
        y_pred,
        linestyle="--",
        color="black",
    )
    ax.set_xscale("log")
    ax.set_xlabel("FLOPs")
    ax.set_ylabel(loss_column)
    if minimization:
        ax.set_yscale("log")
    fig.savefig(save_path / f"{loss_column}.pdf".replace("/", "_"))

In [ ]:
tabpfn_tabarena_baseline

In [ ]:
df.columns

In [ ]:
# Scaling laws for

# max_cells

# N

# D
sns.set_context("talk")
viz_columns = ["parameters", "prior_max_cells", "total_cells", "config/num_datapoints_max", "config/max_features"]

save_path = base_figure_path / "pareto_fitting"
save_path.mkdir(parents=True, exist_ok=True)

combined_figures = {column: plt.subplots(layout='constrained') for column in viz_columns}
slopes = {}

for loss_idx, loss_column in enumerate(loss_columns):

    maximization = "roc_auc" in loss_column or "accuracy" in loss_column
    minimization = not maximization
    pareto = get_pareto_frontier(df, flops_column, loss_column, minimization=minimization)

    for trend_column in viz_columns:
        fig, ax = plt.subplots(layout='constrained')
        fig_c, ax_c = combined_figures[trend_column]

        # same colour for the scatter and its fit, so the single-metric figures and the
        # combined ones use one consistent code (warm/solid = real data, cool/dashed = prior val)
        ax.scatter(pareto[flops_column], pareto[trend_column], marker='x', color=LOSS_STYLES[loss_column]["color"])
        # ax_c.scatter(pareto[flops_column], pareto[trend_column], marker='x')

        slope, intercept, r_value, p_value, std_err = fit_linear_model(np.log(pareto[flops_column]), np.log(pareto[trend_column]))

        ax.plot(FLOP_VALUES, np.exp(intercept + np.log(FLOP_VALUES) * slope),
                **loss_style(loss_column, linestyle="--", linewidth=2))
        ax_c.plot(FLOP_VALUES, np.exp(intercept + np.log(FLOP_VALUES) * slope),
                  **loss_style(loss_column, linewidth=2))

        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("FLOPs")
        ax.set_ylabel(trend_column)

        fig.savefig(save_path / f"{loss_column}-{trend_column}.pdf".replace("/", "_"))
        plt.close(fig)
        slopes[loss_column] = slopes.get(loss_column, {})
        slopes[loss_column][trend_column] = slope

for trend_column, (fig, ax) in combined_figures.items():
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("FLOPs")
    ax.set_ylabel(trend_column)
    family_legends(ax)
    fig.savefig(save_path / f"{trend_column}_c.pdf".replace("/", "_"))


In [ ]:
pareto.columns

In [ ]:
slopes

for key, values in slopes.items():
    fig, ax = plt.subplots(layout='constrained')
    for subkey, value in values.items():
        print(f"  {subkey}: {value:.4f}")
        # x values between 0 and 1000
        x = np.arange(1, 10)
        ax.plot(x, x ** value,label=subkey)
    ax.loglog()
    ax.legend()
    ax.set_xlabel("Compute Multiple")
    ax.set_xlabel("FLOP Multiple")
    fig.suptitle(f"Scaling Slopes for {key}")

    fig.savefig(save_path / f"{key}_slopes.pdf".replace("/", "_"))


In [ ]:
save_path = base_figure_path / "prior_ablation"
save_path.mkdir(parents=True, exist_ok=True)

small_prior = {
    "name": "small prior",
    "marker": "o",
    "config/max_features": 4,
    "config/num_datapoints_max": 100,
}
medium_prior = {
    "name": "medium prior",
    "marker": "s",
    "config/max_features": 8,
    "config/num_datapoints_max": 200,
}
big_prior = {
    "name": "big prior",
    "marker": "^",
    "config/max_features": 16,
    "config/num_datapoints_max": 400,
}

all_data_prior = {
    "name": "best prior / flops",
    "marker": "^",
}
eval_settings = [small_prior, big_prior, all_data_prior]

for loss_idx, loss_column in enumerate(loss_columns):
    fig, ax = plt.subplots(layout='constrained')
    maximization = "roc_auc" in loss_column or "accuracy" in loss_column
    minimization = not maximization


    for eval_setting in eval_settings:
        if True: # minimization:
            initial_grid = initial_grid_min
            bounds = None
            func_form = functional_form_L0

        sigmoid_flag = False
        if loss_column == "real_data/roc_auc" and eval_setting["name"] == "big prior":
            # S-shaped: a sharp logistic step plus a broad one, fitted in ROC AUC space on the big-prior data only
            sigmoid_flag = True
            initial_grid = initial_grid_double_sigmoid
            bounds = bounds_double_sigmoid
            func_form = double_sigmoid_functional_form

        eval_setting = eval_setting.copy()
        label = eval_setting.pop("name")
        marker = eval_setting.pop("marker")
        setting_df = df.copy()
        setting_extrapolation_df = extrapolation_df.copy()
        for column, value in eval_setting.items():
            setting_df = setting_df[setting_df[column]==value]
            setting_extrapolation_df = setting_extrapolation_df[setting_extrapolation_df[column]==value]

        pareto = get_pareto_frontier(setting_df, flops_column, loss_column, minimization=minimization)
        # best run per FLOP level above FILTER_FLOPS_MAX, not used for fitting
        held_out = get_optimal_values(setting_extrapolation_df, flops_column, loss_column, minimization=minimization)

        points = ax.scatter(pareto[flops_column], pareto[loss_column], label=label, marker=marker)
        if len(held_out):
            ax.scatter(held_out[flops_column], held_out[loss_column], marker=marker, facecolors="none",
                       edgecolors=points.get_facecolor(), label=f"{label} (held out)")

        X_data = pareto[flops_column].values
        y_data = np.log(pareto[loss_column].values) if minimization else np.log(1-pareto[loss_column].values)
        if sigmoid_flag:
            y_data = pareto[loss_column].values

        best_params, best_loss = fit_parametric_form(
            func_form=func_form,
            X_data=X_data,
            y_data=y_data,
            initial_grid=list(product(*initial_grid)),
            bounds=bounds,
            delta=1e-3,
            use_scipy=True,
        )

        y_pred = func_form(FLOP_VALUES, best_params, return_log_loss=False)
        y_pred = y_pred if minimization else 1 - y_pred
        if sigmoid_flag:
            y_pred = 1-y_pred

        ax.plot(
            FLOP_VALUES,
            y_pred,
            linestyle="--",
        )
    ax.set_xscale("log")
    ax.set_xlabel("FLOPs")
    ax.set_ylabel(loss_column)
    ax.legend()
    if minimization:
        ax.set_yscale("log")
    fig.savefig(save_path / f"{loss_column}.pdf".replace("/", "_"))

In [ ]:
# I need the roc auc data for big prior
loss_column = "real_data/roc_auc"

eval_setting = big_prior.copy()
label = eval_setting.pop("name")
marker = eval_setting.pop("marker")

setting_df = df.copy()
for column, value in eval_setting.items():
    setting_df = setting_df[setting_df[column]==value]

pareto = get_pareto_frontier(setting_df, flops_column, loss_column, minimization=False)

plt.scatter(pareto[flops_column], pareto[loss_column])
plt.xscale("log")
# save pareto
pareto.to_csv(base_output_path  / f"{loss_column}_big_prior_pareto.csv".replace("/", "_"))

In [ ]:
save_path

In [ ]:
save_path = base_figure_path / "prior_ablation" / "neeratyoy"
save_path.mkdir(parents=True, exist_ok=True)

small_prior = {
    "name": "small prior",
    "marker": "o",
    "config/max_features": 4,
    "config/num_datapoints_max": 100,
}
medium_prior = {
    "name": "medium prior",
    "marker": "s",
    "config/max_features": 8,
    "config/num_datapoints_max": 200,
}
big_prior = {
    "name": "big prior",
    "marker": "^",
    "config/max_features": 16,
    "config/num_datapoints_max": 400,
}

all_data_prior = {
    "name": "tuned prior",
    "marker": "x",
}
eval_settings = [small_prior, medium_prior]

for loss_idx, loss_column in enumerate(loss_columns):
    fig, ax = plt.subplots(layout='constrained')
    maximization = "roc_auc" in loss_column or "accuracy" in loss_column
    minimization = not maximization

    for eval_setting in eval_settings:
        eval_setting = eval_setting.copy()
        label = eval_setting.pop("name")
        marker = eval_setting.pop("marker")
        setting_df = df.copy()
        for column, value in eval_setting.items():
            setting_df = setting_df[setting_df[column]==value]

        if minimization:
            setting_df = setting_df.loc[setting_df.groupby(by="parameters")[loss_column].idxmin()]

        else:
            setting_df = setting_df.loc[setting_df.groupby(by="parameters")[loss_column].idxmax()]
        setting_df.sort_values(by="parameters", ascending=True, inplace=True)
        # pareto = get_pareto_frontier(setting_df, "parameters", loss_column, minimization=minimization)
        pareto = setting_df
        ax.scatter(pareto["parameters"], pareto[loss_column], label=label, marker=marker)

    ax.set_xscale("log")
    ax.set_xlabel("Model parameters")
    ax.set_ylabel(loss_column)
    ax.legend()
    if minimization:
        ax.set_yscale("log")
    fig.savefig(save_path / f"{loss_column}.pdf".replace("/", "_"))

In [ ]:
(df['parameters'].min())


# Approach

In [ ]:
from scaling.utils import fit_parametric_form, functional_form_chin3
from sklearn.metrics import r2_score

LIMIT_PARAMETERS = 20000.

def fit_chin3(data_X, data_y, dense_grid=False):
    if dense_grid:
        initialization = list(product(
            np.arange(0, 25, 1),
            np.arange(0, 2, 0.1),
            np.arange(0, 25, 1),
            np.arange(0, 2, 0.1),
            np.arange(-1, 1, 0.1)
        ))
    else:
        initialization = list(product(
            np.arange(0, 5, 1),
            np.arange(0, 2, 0.5),
            np.arange(0, 5, 1),
            np.arange(0, 2, 0.5),
            np.arange(-1, 1, 0.5)
        ))

    best_params, best_loss = fit_parametric_form(
        functional_form_chin3,
        data_X[:,:2],
        data_y,
        initialization,
    )

    y_true = data_y
    y_pred = functional_form_chin3(data_X[:,:2], best_params)
    r2_log = r2_score(y_true, y_pred)
    r2 = r2_score(np.exp(y_true), np.exp(y_pred))

    print("Quality of Fit")
    print("Log scaled R2: ", r2_log)
    print("R2: ", r2)

    return best_params

In [ ]:
# I need to preprocess the data
loss_column = loss_columns[0]
chin3_data = df.loc[df.groupby(by=['parameters', 'total_cells'])[loss_column].idxmin(), ['parameters', 'total_cells', loss_column]]

chin3_data = chin3_data[chin3_data[loss_column]  < 1.0]
# I need to fit the data
X = chin3_data.values[:, :2]
y = np.log(chin3_data.values[:, 2])
# I need to visualize the data

In [ ]:
# best_params = fit_chin3(X, y)


In [ ]:
from matplotlib.colors import LogNorm

fig, ax = plt.subplots(1, 1, figsize=(6, 4.75), layout='constrained')

# get the range of the x_values
D_contour = np.linspace(df["total_cells"].min()*0.85, df["total_cells"].max()*1.15, 3000)
N_contour = np.linspace(df["parameters"].min()*0.85, df["parameters"].max()*0.5, 3000)

N_cg , D_cg = np.meshgrid(N_contour, D_contour)
L_cg = np.exp(functional_form_chin3(
            np.column_stack((
                N_cg.reshape(-1),
                D_cg.reshape(-1)
            )),
            best_params
)).reshape(N_cg.shape)
cs = ax.contourf(N_cg, D_cg, L_cg, levels=300, cmap="Blues", zorder=-1, norm=LogNorm(vmin=L_cg.min(), vmax=L_cg.max()))

fig.colorbar(cs, ax=ax, label=loss_column)
# ax.set_title("SZP")
ax.set_xlabel("Parameters")
ax.set_ylabel("Total Cells")
ax.loglog()